# 02 — LLM Provider Comparison

Compares LLM providers for agent mobility decisions:
- **Ollama + llama3.1:8b** (local)
- **vLLM + llama3.1:8b** (local server)
- **OpenAI gpt-4o-mini** (API)
- **OpenAI gpt-4o** (API)

**Metrics**:
- **Latency**: time-to-first-token + total response time
- **Humanistic accuracy**: archetype-appropriate destination selection
- **Cost**: tokens / decision, estimated $/1000 decisions
- **JSON compliance**: structured output reliability

In [ ]:
import asyncio
import json
import time
import os
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import AsyncOpenAI

sns.set_theme(style='whitegrid', palette='muted')

with open('data/sample_agents.json') as f:
    AGENTS = json.load(f)

PROVIDERS = {
    'ollama': {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'llama3.1:8b'},
    'vllm':   {'base_url': 'http://localhost:8000/v1',  'api_key': 'vllm',   'model': 'meta-llama/Llama-3.1-8B-Instruct'},
    'gpt4o-mini': {'base_url': None, 'api_key': os.getenv('OPENAI_API_KEY', ''), 'model': 'gpt-4o-mini'},
    'gpt4o':  {'base_url': None, 'api_key': os.getenv('OPENAI_API_KEY', ''), 'model': 'gpt-4o'},
}

In [ ]:
MOBILITY_PROMPT_TEMPLATE = """You are a {archetype} in Barcelona Eixample.
Current needs: hunger={hunger:.1f}, energy={energy:.1f}, social={social:.1f}
Candidate destinations (name, type, distance_m):
{candidates}

Choose the most appropriate destination for your archetype and needs.
Reply with JSON only: {{\"destination\": \"<name>\", \"type\": \"<type>\", \"reasoning\": \"<brief>\"}}"""

SAMPLE_CANDIDATES = [
    ('Bar Montana', 'cafe', 45),
    ('Farmacia Roca', 'pharmacy', 80),
    ('Parc de la Ciutadella', 'park', 150),
    ('Mercado de la Sagrada Familia', 'supermarket', 200),
    ('Sagrada Familia', 'attraction', 300)
]

def build_prompt(agent):
    cands = '\n'.join(f'  - {n} ({t}, {d}m)' for n, t, d in SAMPLE_CANDIDATES)
    return MOBILITY_PROMPT_TEMPLATE.format(
        archetype=agent['archetype'],
        hunger=agent['needs']['hunger'],
        energy=agent['needs']['energy'],
        social=agent['needs']['social'],
        candidates=cands
    )

async def bench_provider(name, config, agents, runs_per_agent=3):
    kwargs = {'api_key': config['api_key']}
    if config['base_url']:
        kwargs['base_url'] = config['base_url']
    client = AsyncOpenAI(**kwargs)
    results = []
    for agent in agents[:5]:  # first 5 agents for speed
        prompt = build_prompt(agent)
        for _ in range(runs_per_agent):
            t0 = time.perf_counter()
            try:
                resp = await client.chat.completions.create(
                    model=config['model'],
                    messages=[{'role': 'user', 'content': prompt}],
                    max_tokens=150, temperature=0.7
                )
                elapsed = (time.perf_counter() - t0) * 1000
                content = resp.choices[0].message.content or ''
                try:
                    parsed = json.loads(content)
                    json_ok = True
                    dest_type = parsed.get('type', '')
                except Exception:
                    json_ok = False
                    dest_type = ''
                # Humanistic accuracy: destination type in expected_destinations
                accurate = dest_type in agent.get('expected_destinations', [])
                in_tokens = resp.usage.prompt_tokens if resp.usage else 0
                out_tokens = resp.usage.completion_tokens if resp.usage else 0
                results.append({'provider': name, 'latency_ms': elapsed,
                                'json_ok': json_ok, 'accurate': accurate,
                                'in_tokens': in_tokens, 'out_tokens': out_tokens})
            except Exception as e:
                results.append({'provider': name, 'latency_ms': None,
                                'json_ok': False, 'accurate': False,
                                'in_tokens': 0, 'out_tokens': 0})
    return results

print('Run: await bench_all_providers() to execute benchmarks')
print('Requires Ollama/vLLM running locally and/or OPENAI_API_KEY set')

In [ ]:
# Run this cell to execute all benchmarks
# async def bench_all():
#     all_results = []
#     for name, config in PROVIDERS.items():
#         print(f'Benchmarking {name}...')
#         r = await bench_provider(name, config, AGENTS)
#         all_results.extend(r)
#         print(f'  done: {len(r)} samples')
#     return pd.DataFrame(all_results)
#
# df = await bench_all()
# df.to_csv('results_02_llm_raw.csv', index=False)

# Placeholder for offline viewing
import numpy as np
np.random.seed(42)
df = pd.DataFrame([
    {'provider': p, 'latency_ms': lat, 'json_ok': True, 'accurate': acc, 'in_tokens': 180, 'out_tokens': 50}
    for p, lat, acc in [
        ('ollama', 1800 + np.random.randn()*300, 0.72),
        ('vllm',   650  + np.random.randn()*100, 0.72),
        ('gpt4o-mini', 450 + np.random.randn()*80, 0.85),
        ('gpt4o', 900 + np.random.randn()*150, 0.93),
    ] for _ in range(15)
])
print(df.groupby('provider')[['latency_ms', 'accurate']].mean())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Violin — latency distribution
sns.violinplot(data=df, x='provider', y='latency_ms', ax=axes[0], inner='box')
axes[0].set_title('Response Latency by Provider')
axes[0].set_ylabel('Latency (ms)')
axes[0].tick_params(axis='x', rotation=15)

# Bar — humanistic accuracy
acc = df.groupby('provider')['accurate'].mean() * 100
acc.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Humanistic Accuracy (%)')
axes[1].set_ylabel('Archetype-Appropriate Decisions (%)')
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis='x', rotation=15)

# Scatter — latency vs accuracy trade-off
summary = df.groupby('provider').agg({'latency_ms': 'median', 'accurate': 'mean'}).reset_index()
axes[2].scatter(summary['latency_ms'], summary['accurate'] * 100, s=120)
for _, row in summary.iterrows():
    axes[2].annotate(row['provider'], (row['latency_ms'], row['accurate']*100),
                     textcoords='offset points', xytext=(5, 5))
axes[2].set_title('Latency vs Accuracy Trade-off')
axes[2].set_xlabel('Median Latency (ms)')
axes[2].set_ylabel('Accuracy (%)')

plt.tight_layout()
plt.savefig('results_02_llm_comparison.png', dpi=150)
plt.show()